#Telecom Domain Read & Write Ops Hackathon - Building Datalake & Lakehouse
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object

##1. Write SQL statements to create:
1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol
4. Using dbutils.fs.mkdirs, create folders:<br>
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):<br>
a. Volume vs DBFS/FileStore<br>
b. Why production teams prefer Volumes for regulated data<br>

In [0]:
%sql
create catalog if not exists telecom_catalog_assign;
create schema if not exists telecom_catalog_assign.landing_zone;
create volume if not exists telecom_catalog_assign.landing_zone.landing_vol;


##Data files to use in this usecase:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

In [0]:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

##2. Filesystem operations
1. Write dbutils.fs code to copy the above datasets into your created Volume folders:
Customer → /Volumes/.../customer/
Usage → /Volumes/.../usage/
Tower (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/

2. Write a command to validate whether files were successfully copied

In [0]:
#creating csv file using "put" function by reaing the data declared in above code
path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"
dbutils.fs.put(f"{path}/customer.csv", customer_csv, True)
dbutils.fs.put(f"{path}/usage.csv", usage_tsv, True)
dbutils.fs.put(f"{path}/tower_logs_region1.csv", tower_logs_region1, True)
dbutils.fs.put(f"{path}/tower_logs_region2.csv", tower_logs_region1, True)
#dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer.csv", customer_csv, overwrite=True)



In [0]:
%fs head /Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer.csv

In [0]:
#Reading the file in simple read function
df = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer.csv", header=False)
#.toDF("Cid","Name","age","city","plan")
#df.show()
display(df)

In [0]:
dbutils.fs.ls(path)


##3. Spark Directory Read Use Cases
1. Read all tower logs using:
Path glob filter (example: *.csv)
Multiple paths input
Recursive lookup

2. Demonstrate these 3 reads separately:
Using pathGlobFilter
Using list of paths in spark.read.csv([path1, path2])
Using .option("recursiveFileLookup","true")

3. Compare the outputs and understand when each should be used.

In [0]:
#Read multiple csv files at one shot using *
csv_df4=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_*",inferSchema=True,header=True,sep='|')
#csv_df4.count() 
display(csv_df4)

In [0]:
# Using recursiveFileLookup
df_multiple_sources=spark.read.csv(path=["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region2.csv"],inferSchema=True,header=True,sep='|',recursiveFileLookup=False)
#inferSchema=True,header=True,sep=',',pathGlobFilter="custs_header_*",recursiveFileLookup=True)
#.toDF("cid","fn","ln","a","p")
print(df_multiple_sources.count())
display(df_multiple_sources)

In [0]:
option_read_df2=spark.read.option("header","True").option("inferSchema","true").option("sep","~").option("recursiveFileLookup","True").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv")
option_read_df2.show(2)

##4. Schema Inference, Header, and Separator
1. Try the Customer, Usage files with the option and options using read.csv and format function:<br>
header=false, inferSchema=false<br>
or<br>
header=true, inferSchema=true<br>
2. Write a note on What changed when we use header or inferSchema  with true/false?<br>
3. How schema inference handled “abc” in age?<br>

In [0]:
# Reading multiple path files in one shot and using inferSchema = False & header=True
csv_df1=spark.read.csv(path=["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region2.csv"],inferSchema=False,header=True,sep='|')
#csv_df1=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv", header=False)
display(csv_df1)

In [0]:
# Reading multiple path files in one shot and using inferSchema =True & header=True
csv_df1=spark.read.csv(path=["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region2.csv"],inferSchema=True,header=True,sep='|')
#csv_df1=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv", header=False)
display(csv_df1)

In [0]:
# Reading csv files using "option"

csv_df2=spark.read.option("header","True").option("inferSchema","true").option("sep","|").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv")
display(csv_df2)

In [0]:
#Reading csv function using  "options"
csv_df3=spark.read.options(header="True",inferSchema="true",sep="|").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv")
display(csv_df3)

##5. Column Renaming Usecases
1. Apply column names using string using toDF function for customer data
2. Apply column names and datatype using the schema function for usage data
3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data 

In [0]:
# Apply column names using string using toDF function for customer data
customer_df1 = spark.read.options(
    header="false",
    inferSchema="false"
    ).format('csv').load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer.csv").toDF("customer_id","first_name","age","city","plan_type")
display(customer_df1)

In [0]:
sample_usage_df = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage.csv",inferSchema=True,header=True,sep='\t')
display(sample_usage_df)

In [0]:
#Apply column names and datatype using the schema function for usage data
str_struct="customer_id integer,voice_mins integer,data_mb integer,sms_count integer"
csv_df2 = spark.read.schema(str_struct).option("header", "true").option("sep", "\t").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage.csv")
display(csv_df2)

In [0]:
sample_tower_df = spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv",inferSchema=True,header=True,sep='|')
display(sample_tower_df)

In [0]:
#Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data


from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, StringType
custom_schema = StructType([
    StructField("event_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("tower_id", StringType(), True),
    StructField("signal_strength", IntegerType(), True),
    StructField("timestamp", TimestampType(), True)
])

csv_df3 = spark.read.schema(custom_schema).option("header", "true").option("sep", "|").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_logs_region1.csv")
display(csv_df3)



## Spark Write Operations using 
- csv, json, orc, parquet, delta, saveAsTable, insertInto, xml with different write mode, header and sep options

##6. Write Operations (Data Conversion/Schema migration) – CSV Format Usecases
1. Write customer data into CSV format using overwrite mode
2. Write usage data into CSV format using append mode
3. Write tower data into CSV format with header enabled and custom separator (|)
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
#Write customer data into CSV format using overwrite mode

#Write usage data into CSV format using append mode
#Write tower data into CSV format with header enabled and custom separator (|)
#Read the tower data in a dataframe and show only 5 rows.
#Download the file into local from the catalog volume location and see the data of any of the above files opening in a notepad++.

##7. Write Operations (Data Conversion/Schema migration)– JSON Format Usecases
1. Write customer data into JSON format using overwrite mode
2. Write usage data into JSON format using append mode and snappy compression format
3. Write tower data into JSON format using ignore mode and observe the behavior of this mode
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##8. Write Operations (Data Conversion/Schema migration) – Parquet Format Usecases
1. Write customer data into Parquet format using overwrite mode and in a gzip format
2. Write usage data into Parquet format using error mode
3. Write tower data into Parquet format with gzip compression option
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##9. Write Operations (Data Conversion/Schema migration) – Orc Format Usecases
1. Write customer data into ORC format using overwrite mode
2. Write usage data into ORC format using append mode
3. Write tower data into ORC format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

##10. Write Operations (Data Conversion/Schema migration) – Delta Format Usecases
1. Write customer data into Delta format using overwrite mode
2. Write usage data into Delta format using append mode
3. Write tower data into Delta format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.
6. Compare the parquet location and delta location and try to understand what is the differentiating factor, as both are parquet files only.

##11. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using saveAsTable() as a managed table
2. Write usage data using saveAsTable() with overwrite mode
3. Drop the managed table and verify data removal
4. Go and check the table overview and realize it is in delta format in the Catalog.
5. Use spark.read.sql to write some simple queries on the above tables created.


##12. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using insertInto() in a new table and find the behavior
2. Write usage data using insertTable() with overwrite mode

##13. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data into XML format using rowTag as cust
2. Write usage data into XML format using overwrite mode with the rowTag as usage
3. Download the xml data and open the file in notepad++ and see how the xml file looks like.

##14. Compare all the downloaded files (csv, json, orc, parquet, delta and xml) 
1. Capture the size occupied between all of these file formats and list the formats below based on the order of size from small to big.

###15. Try to do permutation and combination of performing Schema Migration & Data Conversion operations like...
1. Read any one of the above orc data in a dataframe and write it to dbfs in a parquet format
2. Read any one of the above parquet data in a dataframe and write it to dbfs in a delta format
3. Read any one of the above delta data in a dataframe and write it to dbfs in a xml format
4. Read any one of the above delta table in a dataframe and write it to dbfs in a json format
5. Read any one of the above delta table in a dataframe and write it to another table

##16. Do a final exercise of defining one/two liner of... 
1. When to use/benifits csv
2. When to use/benifits json
3. When to use/benifit orc
4. When to use/benifit parquet
5. When to use/benifit delta
6. When to use/benifit xml
7. When to use/benifit delta tables
